## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of LangGraph for orchestrating complex agentic RAG workflows.
* Implement advanced RAG patterns such as Corrective RAG, Self-RAG, and adaptive query routing.
* Design and integrate self-correcting search and retrieval tools within a multi-agent system.
* Apply state management and conditional routing effectively to build robust and dynamic RAG applications.
* Evaluate and refine agentic RAG system performance based on real-world query scenarios.


## Final Assessment: Building Advanced Agentic RAG Systems

Welcome to the final assessment for ADV-04: Building Agentic RAG Systems with LangGraph! This assessment is designed to evaluate your mastery of the core concepts and practical skills developed throughout this course. As an AI search engineer, your ability to design, implement, and debug sophisticated retrieval-augmented generation (RAG) systems is paramount in 2026.

This assessment will challenge you to integrate various advanced RAG patterns, including Corrective RAG, Self-RAG, adaptive query routing, and self-correcting search tools, all orchestrated using the powerful LangGraph framework. You will demonstrate your understanding of state management, conditional logic, and tool integration to build a resilient and intelligent RAG agent.

The assessment is structured into two main parts:

1.  **Review Questions**: A set of conceptual and theoretical questions to test your understanding of the underlying principles and design choices in agentic RAG.
2.  **Capstone Coding Problem**: A practical, hands-on coding challenge where you will build a sophisticated agentic RAG system from scratch, applying all the techniques learned in the course.

Your goal is to showcase your ability to not only recall information but also to apply it creatively to solve complex, real-world problems in the domain of advanced retrieval QA. Good luck!


### Review Questions

Answer the following questions concisely, demonstrating your understanding of the concepts discussed in this course.

1.  **LangGraph State Management**: Explain the role of `StateGraph` and `State` in LangGraph. How does the `State` object facilitate communication and data flow between different nodes in a graph? Provide an example of a custom `State` object that would be useful in an agentic RAG system.

2.  **Adaptive Query Routing**: Describe a scenario where adaptive query routing is crucial for an agentic RAG system. How would you implement a routing mechanism in LangGraph that decides between an internal knowledge base search and an external web search based on the user's query?

3.  **Corrective RAG vs. Self-RAG**: Differentiate between Corrective RAG and Self-RAG. Provide a concrete example of when you would choose to implement one over the other in a production RAG system.

4.  **Self-Correcting Search Tools**: How can an LLM be used to create a "self-correcting" search tool? Outline the steps an agentic RAG system would take if an initial search query yields irrelevant results, and how the LLM facilitates correction.

5.  **Conditional Edges**: Explain the significance of conditional edges in LangGraph. Provide a Pythonic example of how a conditional edge might be defined to transition between an `answer_generation` node and a `refinement_loop` node based on an `answer_confidence` score.

6.  **Tool Invocation in LangGraph**: Describe the typical pattern for integrating external tools (e.g., a vector database retriever, a web search API) into a LangGraph agent. What are the key considerations for making these tools robust and reliable within an agentic workflow?

7.  **Handling Ambiguity**: How would you design an agentic RAG system to handle ambiguous user queries? Discuss at least two strategies involving LangGraph and its components to clarify or refine such queries before attempting retrieval.


### Capstone Coding Problem: Enterprise Knowledge Navigator with Self-Correction

**Scenario:**

You are tasked with building an advanced enterprise knowledge navigation system for a large tech company. This system needs to answer complex questions from employees, drawing information from both an internal, curated knowledge base (simulated by a local document store) and the broader internet (simulated by a web search API). The key requirement is that the system must be highly reliable, capable of self-correction, and able to adapt its strategy based on the nature of the query and the quality of initial results.

**Your Task:**

Implement an agentic RAG system using LangGraph that can perform the following:

1.  **Initial Query Analysis & Routing**: Upon receiving a user query, an LLM should analyze it to determine if it's primarily an internal knowledge base question or if it requires external web search. It should then route the query to the appropriate tool.
2.  **Tool Execution**: Execute either the `internal_kb_retriever` (for internal documents) or the `web_search_tool` (for external information).
3.  **Answer Generation**: Generate an initial answer based on the retrieved context.
4.  **Answer Grading & Confidence Check**: An LLM should then grade the generated answer for relevance, completeness, and confidence. If the confidence is below a certain threshold, or if the answer is deemed irrelevant/incomplete, the system should trigger a self-correction loop.
5.  **Self-Correction Loop (Corrective RAG/Self-RAG)**:
    *   If the initial answer is poor, the system should attempt to **rewrite the original query** to be more precise or to explore a different angle.
    *   It should then **re-route** the rewritten query, potentially trying the *other* tool (e.g., if it initially used internal KB, now try web search, or vice-versa, or retry the same tool with a better query).
    *   This loop can repeat a maximum of `N` times (e.g., 2-3 times) to prevent infinite loops.
6.  **Final Answer**: Once a high-confidence answer is generated or the maximum correction attempts are reached, the system should output the best available answer.

**Tools Provided (or to be Mocked):**

*   `internal_kb_retriever(query: str) -> List[str]`: Simulates retrieving relevant chunks from an internal knowledge base.
*   `web_search_tool(query: str) -> List[str]`: Simulates performing a web search and returning snippets.
*   `llm_grader(query: str, answer: str, context: List[str]) -> Dict[str, Any]`: An LLM call that grades the answer, returning a confidence score and a boolean indicating if correction is needed.
*   `llm_query_rewriter(original_query: str, previous_context: List[str], previous_answer: str) -> str`: An LLM call that rewrites the query for a better search/retrieval attempt.
*   `llm_router(query: str) -> str`: An LLM call that decides between 'internal_kb' or 'web_search'.

**Constraints:**

*   Use LangGraph for orchestration.
*   Implement a clear `State` object to manage the workflow.
*   Include at least one self-correction loop.
*   Handle a maximum number of correction attempts.
*   Provide example queries to demonstrate the system's capabilities.

---


In [ ]:
import os
from typing import List, Dict, Any, TypedDict, Optional

# Mock LLM and Tool implementations for the starter template
class MockLLM:
    def invoke(self, prompt: str) -> str:
        # Simulate LLM response
        if "route" in prompt.lower():
            return "internal_kb" # Default route for starter
        elif "grade" in prompt.lower():
            return "{'confidence': 0.6, 'needs_correction': True}" # Default to needing correction
        elif "rewrite" in prompt.lower():
            return "rewritten query for better search"
        else:
            return "Generated answer based on context."

class MockInternalKBRetriever:
    def invoke(self, query: str) -> List[str]:
        print(f"[Internal KB] Retrieving for: {query}")
        if "langgraph" in query.lower():
            return ["LangGraph is a library for building stateful, multi-actor applications with LLMs.", "It extends LangChain to enable cyclical computation."]
        return [f"Internal document about {query} (placeholder)."]

class MockWebSearchTool:
    def invoke(self, query: str) -> List[str]:
        print(f"[Web Search] Searching for: {query}")
        if "agentic rag" in query.lower():
            return ["Agentic RAG systems combine LLMs with external tools for dynamic information retrieval.", "They often involve self-correction and adaptive routing."]
        return [f"Web search result for {query} (placeholder)."]

# Initialize mock tools and LLM
llm = MockLLM()
internal_kb_retriever = MockInternalKBRetriever()
web_search_tool = MockWebSearchTool()

# --- Define the Graph State ---
# This defines the object that holds the state of our graph
class GraphState(TypedDict):
    """Represents the state of our graph."""
    query: str
    context: List[str]
    answer: str
    route: Optional[str]
    confidence: float
    attempts: int

# --- Define the Nodes ---
# Placeholder functions for the nodes. You will implement the logic.

def route_query(state: GraphState) -> Dict[str, Any]:
    """Determines whether to use internal KB or web search."""
    print("---ROUTE QUERY---")
    query = state["query"]
    # TODO: Implement LLM-based routing logic here
    # Example: llm_router.invoke(query)
    # For now, just return a default route
    return {"route": "internal_kb"} # Placeholder

def retrieve_internal_kb(state: GraphState) -> Dict[str, Any]:
    """Retrieves documents from the internal knowledge base."""
    print("---RETRIEVE INTERNAL KB---")
    query = state["query"]
    # TODO: Invoke internal_kb_retriever
    # For now, return mock context
    return {"context": internal_kb_retriever.invoke(query)}

def web_search(state: GraphState) -> Dict[str, Any]:
    """Performs a web search."""
    print("---WEB SEARCH---")
    query = state["query"]
    # TODO: Invoke web_search_tool
    # For now, return mock context
    return {"context": web_search_tool.invoke(query)}

def generate_answer(state: GraphState) -> Dict[str, Any]:
    """Generates an answer based on the retrieved context."""
    print("---GENERATE ANSWER---")
    query = state["query"]
    context = state["context"]
    # TODO: Implement LLM-based answer generation
    # For now, return a placeholder answer
    return {"answer": f"Answer for '{query}' based on context: {context[0] if context else 'No context.'}"}

def grade_answer(state: GraphState) -> Dict[str, Any]:
    """Grades the generated answer and decides if correction is needed."""
    print("---GRADE ANSWER---")
    query = state["query"]
    answer = state["answer"]
    context = state["context"]
    # TODO: Implement LLM-based grading logic
    # Example: llm_grader.invoke(query, answer, context)
    # For now, simulate a grade
    current_attempts = state.get("attempts", 0)
    if current_attempts < 1: # Force correction on first attempt
        return {"confidence": 0.4, "needs_correction": True}
    else:
        return {"confidence": 0.8, "needs_correction": False}

def rewrite_query(state: GraphState) -> Dict[str, Any]:
    """Rewrites the query for a better search/retrieval attempt."""
    print("---REWRITE QUERY---")
    original_query = state["query"]
    previous_context = state["context"]
    previous_answer = state["answer"]
    # TODO: Implement LLM-based query rewriting
    # Example: llm_query_rewriter.invoke(original_query, previous_context, previous_answer)
    # For now, return a simple rewritten query
    return {"query": f"Refined query for: {original_query}", "attempts": state.get("attempts", 0) + 1}

# --- Define the Edges (Conditional Logic) ---
# You will define the conditional logic for transitions here.

def decide_to_correct(state: GraphState) -> str:
    """Decides whether to proceed to final answer or enter correction loop."""
    print("---DECIDE TO CORRECT---")
    confidence = state.get("confidence", 0.0)
    needs_correction = state.get("needs_correction", False)
    attempts = state.get("attempts", 0)
    MAX_ATTEMPTS = 2

    if needs_correction and attempts < MAX_ATTEMPTS:
        print(f"Low confidence ({confidence}), attempting correction. Attempt {attempts+1}/{MAX_ATTEMPTS}")
        return "rewrite_query"
    else:
        print(f"High confidence ({confidence}) or max attempts reached. Finishing.")
        return "final_answer"

def route_after_rewrite(state: GraphState) -> str:
    """After rewriting, re-route the query."""
    print("---ROUTE AFTER REWRITE---")
    # TODO: Implement logic to re-route, potentially trying the other tool
    # For now, just re-route to internal KB for simplicity
    return "internal_kb_retrieval"

# --- Build the LangGraph ---
from langgraph.graph import StateGraph, END

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("route_query", route_query)
workflow.add_node("internal_kb_retrieval", retrieve_internal_kb)
workflow.add_node("web_search_retrieval", web_search)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("grade_answer", grade_answer)
workflow.add_node("rewrite_query", rewrite_query)

# Set entry point
workflow.set_entry_point("route_query")

# Add edges (you will fill these in based on your routing logic)
# Example: workflow.add_edge("route_query", "internal_kb_retrieval")
# Example: workflow.add_conditional_edges("grade_answer", decide_to_correct, {"rewrite_query": "rewrite_query", "final_answer": END})

# TODO: Define the full graph structure with conditional edges for routing and self-correction.
# Remember to handle the transitions from 'route_query' to 'internal_kb_retrieval' or 'web_search_retrieval'
# and the loop back from 'rewrite_query' to a retrieval step.

# Compile the graph (after defining all nodes and edges)
# app = workflow.compile()

# Example usage (will not work until graph is fully defined):
# initial_state = {"query": "What is LangGraph?", "context": [], "answer": "", "route": None, "confidence": 0.0, "attempts": 0}
# for s in app.stream(initial_state):
#     print(s)

print("Starter template loaded. Your task is to complete the LangGraph definition and implement the detailed logic within the node functions.")


In [ ]:
import os
import json
from typing import List, Dict, Any, TypedDict, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools import DuckDuckGoSearchRun # Using a real search tool for better demo
from langchain_openai import ChatOpenAI # Assuming OpenAI API key is available
from langgraph.graph import StateGraph, END

# --- Configuration and API Keys ---
# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# --- Initialize LLM ---
# Using a modern LLM, e.g., GPT-4o or a similar model from 2026
llm = ChatOpenAI(model="gpt-4o-2024-05-13", temperature=0.0)

# --- Define Tools ---
# 1. Internal KB Retriever (Mocked for simplicity, but could be a real vector DB)
class InternalKBRetriever:
    def invoke(self, query: str) -> List[str]:
        print(f"[Internal KB] Retrieving for: {query}")
        # Simulate a more sophisticated retrieval based on keywords
        if "langgraph" in query.lower() or "agentic" in query.lower():
            return [
                "LangGraph is a library for building stateful, multi-actor applications with LLMs, extending LangChain for cyclical computation.",
                "It's ideal for agentic RAG systems requiring complex orchestration and self-correction."
            ]
        elif "enterprise knowledge" in query.lower() or "company policy" in query.lower():
            return [
                "Our enterprise knowledge base contains policies on remote work, expense reporting, and security protocols.",
                "Access to specific documents requires authentication via our internal portal."
            ]
        return [f"Internal document about {query} (no specific match found, placeholder)."]

internal_kb_retriever = InternalKBRetriever()

# 2. Web Search Tool (Using DuckDuckGoSearchRun for a real-world feel)
web_search_tool = DuckDuckGoSearchRun()

# --- LLM-based Tool Wrappers ---
# These use the LLM to perform specific tasks like routing, grading, and rewriting.

# LLM Router
llm_router_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert router. Analyze the user's query and determine if it's best answered by an 'internal_kb' (company-specific knowledge) or 'web_search' (general internet knowledge). Respond with only 'internal_kb' or 'web_search'."),
    ("human", "{query}")
])
llm_router = llm_router_prompt | llm | StrOutputParser()

# LLM Answer Grader
llm_grader_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert answer grader. Given a user query, a generated answer, and the context used, evaluate the answer's relevance, completeness, and confidence. Respond with a JSON object containing 'confidence' (float, 0.0-1.0) and 'needs_correction' (boolean). If the answer is not directly relevant, incomplete, or seems low confidence, set 'needs_correction' to true."),
    ("human", "Query: {query}\nContext: {context}\nAnswer: {answer}")
])
# Using PydanticOutputParser for structured output from LLM
from langchain_core.pydantic_v1 import BaseModel, Field
class GradeOutput(BaseModel):
    confidence: float = Field(description="Confidence score of the answer (0.0-1.0).")
    needs_correction: bool = Field(description="True if the answer needs correction or refinement.")

llm_grader = llm_grader_prompt | llm.with_structured_output(GradeOutput)

# LLM Query Rewriter
llm_query_rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert query rewriter. Given an original query, previous context, and a previous answer that was deemed insufficient, rewrite the original query to be more precise, explore a different angle, or broaden the search to find better information. Focus on improving the chances of finding a good answer. Respond with only the rewritten query string."),
    ("human", "Original Query: {original_query}\nPrevious Context: {previous_context}\nPrevious Answer: {previous_answer}")
])
llm_query_rewriter = llm_query_rewriter_prompt | llm | StrOutputParser()

# LLM Answer Generator
llm_answer_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Based on the provided context, answer the user's query comprehensively and concisely. If the context does not contain enough information, state that you cannot fully answer based on the provided information."),
    ("human", "Query: {query}\nContext: {context}")
])
llm_answer_generator = llm_answer_generator_prompt | llm | StrOutputParser()

# --- Define the Graph State ---
class GraphState(TypedDict):
    """Represents the state of our graph."""
    query: str # The current query (can be original or rewritten)
    original_query: str # The initial query from the user
    context: List[str] # Retrieved context from tools
    answer: str # Generated answer
    route: Optional[str] # 'internal_kb' or 'web_search'
    confidence: float # Confidence score of the last answer
    attempts: int # Number of correction attempts made
    previous_context: List[str] # Context from previous attempt
    previous_answer: str # Answer from previous attempt

# --- Define the Nodes ---
MAX_ATTEMPTS = 2 # Maximum number of self-correction attempts

def route_query(state: GraphState) -> Dict[str, Any]:
    """Determines whether to use internal KB or web search."""
    print("---NODE: ROUTE QUERY---")
    query = state["query"]
    # Use LLM to decide the route
    route_decision = llm_router.invoke({"query": query})
    print(f"Routing decision: {route_decision}")
    return {"route": route_decision.strip().lower()}

def retrieve_internal_kb(state: GraphState) -> Dict[str, Any]:
    """Retrieves documents from the internal knowledge base."""
    print("---NODE: RETRIEVE INTERNAL KB---")
    query = state["query"]
    context = internal_kb_retriever.invoke(query)
    return {"context": context}

def web_search(state: GraphState) -> Dict[str, Any]:
    """Performs a web search."""
    print("---NODE: WEB SEARCH---")
    query = state["query"]
    # DuckDuckGoSearchRun returns a single string, split by newline for consistency
    search_results = web_search_tool.invoke(query)
    context = search_results.split('\n') if search_results else []
    return {"context": context}

def generate_answer(state: GraphState) -> Dict[str, Any]:
    """Generates an answer based on the retrieved context."""
    print("---NODE: GENERATE ANSWER---")
    query = state["query"]
    context = state["context"]
    answer = llm_answer_generator.invoke({"query": query, "context": "\n".join(context)})
    return {"answer": answer}

def grade_answer(state: GraphState) -> Dict[str, Any]:
    """Grades the generated answer and decides if correction is needed."""
    print("---NODE: GRADE ANSWER---")
    query = state["query"]
    answer = state["answer"]
    context = state["context"]
    
    grade_output = llm_grader.invoke({"query": query, "answer": answer, "context": "\n".join(context)})
    
    print(f"Grade: Confidence={grade_output.confidence}, Needs Correction={grade_output.needs_correction}")
    return {"confidence": grade_output.confidence, "needs_correction": grade_output.needs_correction,
            "previous_context": context, "previous_answer": answer}

def rewrite_query(state: GraphState) -> Dict[str, Any]:
    """Rewrites the query for a better search/retrieval attempt."""
    print("---NODE: REWRITE QUERY---")
    original_query = state["original_query"] # Use original query for context
    previous_context = state["previous_context"]
    previous_answer = state["previous_answer"]
    
    rewritten_query = llm_query_rewriter.invoke({
        "original_query": original_query,
        "previous_context": "\n".join(previous_context),
        "previous_answer": previous_answer
    })
    print(f"Rewritten query: {rewritten_query}")
    return {"query": rewritten_query, "attempts": state.get("attempts", 0) + 1}

# --- Define the Edges (Conditional Logic) ---

def decide_next_step(state: GraphState) -> str:
    """Decides whether to proceed to final answer, rewrite, or try other tool."""
    print("---CONDITIONAL EDGE: DECIDE NEXT STEP---")
    needs_correction = state.get("needs_correction", False)
    attempts = state.get("attempts", 0)
    current_route = state.get("route")

    if not needs_correction or attempts >= MAX_ATTEMPTS:
        print("Decision: Finish (high confidence or max attempts reached).")
        return "final_answer"
    else:
        print(f"Decision: Needs correction (attempt {attempts+1}/{MAX_ATTEMPTS}).")
        # If we need correction, we rewrite the query
        return "rewrite_query"

def route_after_rewrite(state: GraphState) -> str:
    """After rewriting, re-route the query, potentially trying the other tool."""
    print("---CONDITIONAL EDGE: ROUTE AFTER REWRITE---")
    # For simplicity, let's alternate or re-evaluate the route
    # A more advanced system might use an LLM to decide again or keep track of tried routes
    current_route = state.get("route")
    if current_route == "internal_kb":
        print("Re-routing to web_search after rewrite.")
        return "web_search_retrieval"
    else:
        print("Re-routing to internal_kb after rewrite.")
        return "internal_kb_retrieval"

def route_initial_query(state: GraphState) -> str:
    """Routes the initial query based on the 'route' state."""
    print("---CONDITIONAL EDGE: ROUTE INITIAL QUERY---")
    route = state.get("route")
    if route == "internal_kb":
        return "internal_kb_retrieval"
    elif route == "web_search":
        return "web_search_retrieval"
    else:
        # Fallback or error handling
        print(f"Invalid route: {route}. Defaulting to internal_kb.")
        return "internal_kb_retrieval"

# --- Build the LangGraph ---
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("route_query", route_query)
workflow.add_node("internal_kb_retrieval", retrieve_internal_kb)
workflow.add_node("web_search_retrieval", web_search)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("grade_answer", grade_answer)
workflow.add_node("rewrite_query", rewrite_query)

# Set entry point
workflow.set_entry_point("route_query")

# Add edges
# 1. After initial routing, go to appropriate retrieval tool
workflow.add_conditional_edges(
    "route_query",
    route_initial_query, # This function decides the next node based on state["route"]
    {
        "internal_kb_retrieval": "internal_kb_retrieval",
        "web_search_retrieval": "web_search_retrieval"
    }
)

# 2. After retrieval, generate an answer
workflow.add_edge("internal_kb_retrieval", "generate_answer")
workflow.add_edge("web_search_retrieval", "generate_answer")

# 3. After generating an answer, grade it
workflow.add_edge("generate_answer", "grade_answer")

# 4. After grading, decide whether to finish or rewrite the query
workflow.add_conditional_edges(
    "grade_answer",
    decide_next_step,
    {
        "rewrite_query": "rewrite_query",
        "final_answer": END
    }
)

# 5. After rewriting the query, re-route it (potentially to the other tool)
workflow.add_conditional_edges(
    "rewrite_query",
    route_after_rewrite,
    {
        "internal_kb_retrieval": "internal_kb_retrieval",
        "web_search_retrieval": "web_search_retrieval"
    }
)

# Compile the graph
app = workflow.compile()

print("LangGraph agentic RAG system compiled successfully!")

# --- Example Usage ---
print("\n--- Running Example 1: Internal KB Query with Correction ---")
initial_state_1 = {
    "query": "What is LangGraph and how is it used in agentic systems?",
    "original_query": "What is LangGraph and how is it used in agentic systems?",
    "context": [], "answer": "", "route": None, "confidence": 0.0, "attempts": 0,
    "previous_context": [], "previous_answer": ""
}

for s in app.stream(initial_state_1):
    print(s)

final_state_1 = None
for s in app.stream(initial_state_1):
    final_state_1 = s
print(f"\nFinal Answer 1: {final_state_1.get('generate_answer', {}).get('answer', 'No answer generated.')}")
print(f"Final Confidence 1: {final_state_1.get('grade_answer', {}).get('confidence', 0.0)}")
print(f"Total Attempts 1: {final_state_1.get('rewrite_query', {}).get('attempts', 0)}")

print("\n--- Running Example 2: Web Search Query with Correction ---")
initial_state_2 = {
    "query": "What are the latest trends in AI search engineering for 2026?",
    "original_query": "What are the latest trends in AI search engineering for 2026?",
    "context": [], "answer": "", "route": None, "confidence": 0.0, "attempts": 0,
    "previous_context": [], "previous_answer": ""
}

final_state_2 = None
for s in app.stream(initial_state_2):
    final_state_2 = s
print(f"\nFinal Answer 2: {final_state_2.get('generate_answer', {}).get('answer', 'No answer generated.')}")
print(f"Final Confidence 2: {final_state_2.get('grade_answer', {}).get('confidence', 0.0)}")
print(f"Total Attempts 2: {final_state_2.get('rewrite_query', {}).get('attempts', 0)}")

print("\n--- Running Example 3: Simple Query (should pass quickly) ---")
initial_state_3 = {
    "query": "What is our company policy on remote work?",
    "original_query": "What is our company policy on remote work?",
    "context": [], "answer": "", "route": None, "confidence": 0.0, "attempts": 0,
    "previous_context": [], "previous_answer": ""
}

final_state_3 = None
for s in app.stream(initial_state_3):
    final_state_3 = s
print(f"\nFinal Answer 3: {final_state_3.get('generate_answer', {}).get('answer', 'No answer generated.')}")
print(f"Final Confidence 3: {final_state_3.get('grade_answer', {}).get('confidence', 0.0)}")
print(f"Total Attempts 3: {final_state_3.get('rewrite_query', {}).get('attempts', 0)}")
